# Sesión 2 · Ecosistema de modelos y consumo vía API

**AI Engineer I: Fundamentos de IA Generativa** · Módulo 1 · Edutek

---

## 🎯 Objetivos de la sesión

Al terminar este notebook podrás:

1. **Elegir un modelo con criterio** (calidad, costo, privacidad y latencia) y decidir entre *comprar* una API o *construir* tu propio serving.
2. **Llamar a un modelo de forma robusta:** con *streaming*, midiendo latencia (TTFT) y tokens por segundo, y con **reintentos** para los errores transitorios.
3. **Obtener salida estructurada:** pedir JSON, **validarlo** con un esquema y reaccionar si falla.

## 🗺️ Mapa del notebook

| Parte | Tema | ¿Consume dinero? |
| --- | --- | --- |
| 0 | Preparar el entorno | No |
| 1 | El mapa del ecosistema: propietario vs open-weight | No (*) |
| 2 | Anatomía de una llamada | **Sí, centavos** |
| 3 | Streaming y latencia (TTFT, tokens/s) | **Sí, centavos** |
| 4 | Errores y reintentos | No (simulado) |
| 5 | Propietario vs open-weight, cara a cara | **Sí, centavos** (la parte local es gratis) |
| 6 | Cómo elegir un modelo | No |
| 7 | Build vs Buy | No |
| 8 | Salida estructurada (JSON validado) | **Sí, centavos** |
| 9 | Cierre | No |

(*) Listar los modelos disponibles es una consulta sin costo, pero requiere tu API key.

---
# Parte 0 · Preparar el entorno

Verificamos Python, cargamos la API key desde `.env` y traemos nuestro **cliente reutilizable** (`cliente_llm.py`).

## ¿Qué es `cliente_llm.py`?

Es el módulo que construiremos y **usaremos todo el curso**. Vive en esta misma carpeta y agrupa lo que todo cliente serio necesita: streaming con medición, reintentos, conteo de tokens/costo, dos proveedores propietarios intercambiables (Anthropic y OpenAI), uno open-weight local (LM Studio) y salida estructurada validada. Ábrelo en tu editor y léelo: el notebook explica cada pieza, pero el código está ahí.

> **El proveedor se elige con una variable de entorno** (`LLM_PROVIDER=anthropic` u `openai` en tu `.env`): el resto del código no cambia. ¿Usas otro servicio compatible con OpenAI (Groq, OpenRouter…)? Define `OPENAI_BASE_URL`; ver `docs/PROVEEDORES.md`.

In [ ]:
import sys

print("Python:", sys.version.split()[0])
assert sys.version_info[:2] == (3, 12), (
    "Esta sesión está fijada a Python 3.12. Ejecuta `uv sync` dentro de la carpeta "
    "s02-ecosistema-api y selecciona el kernel .venv (ver README)."
)

In [ ]:
import os
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv(usecwd=True))

from cliente_llm import hay_api_key, modelo_economico, proveedor_activo, variable_api_key

PROVEEDOR = proveedor_activo()          # LLM_PROVIDER en tu .env: "anthropic" (por defecto) u "openai"
HAY_KEY = hay_api_key()
print(f"Proveedor: {PROVEEDOR} · modelo económico: {modelo_economico()}")
print(f"¿API key ({variable_api_key()}) detectada?:", "sí ✅" if HAY_KEY else "no ❌ (puedes seguir; se omitirán las celdas de pago)")

### 🔌 Interruptor general de gasto

💰 **FYI:** las celdas de código que llaman a la API de pago de tu proveedor (Anthropic u OpenAI) llevan este emoji en su primera línea, con una estimación de lo que cuestan. Esta variable las controla todas: con `False` no llaman a la API; con `True` y una API key válida, se ejecutan. Te recomendamos tener un límite de gasto mensual configurado en tu cuenta (ver `docs/COSTOS.md`).

In [ ]:
EJECUTAR_LLAMADAS_DE_PAGO = True   # ← ponla en False para no gastar nada
PUEDE_GASTAR = EJECUTAR_LLAMADAS_DE_PAGO and HAY_KEY
print("Celdas de pago activas:", PUEDE_GASTAR)

In [ ]:
import anthropic, openai
from cliente_llm import (
    PRECIOS, Gasto, ResultadoLLM, costo_usd, es_reintentable, con_reintentos,
    llamar_llm, llamar_local, local_disponible, modelo_local, url_local, pedir_json,
    pedir_estructurado_nativo,
)

gasto = Gasto()
MODELO = modelo_economico()     # modelo económico del proveedor elegido, para toda la sesión
print("Módulo cargado. Proveedor:", PROVEEDOR, "· modelo por defecto:", MODELO)

---
# Parte 1 · El mapa del ecosistema

## Dos grandes familias

| | **Propietarios** | **Open-weight** |
| --- | --- | --- |
| Ejemplos de familias | Claude, GPT, Gemini | Llama, Mistral, Qwen |
| Cómo se usan | Por la **API del proveedor** | Publican sus **pesos**; tú los ejecutas donde quieras |
| Costo | Pagas **por token**; no operas nada | No pagas por token, pero **pagas la infraestructura** |
| Control | Lo que permita la API | Lo decides tú (cuantización, ajuste, versiones fijas) |
| Privacidad | Depende del **contrato** con el proveedor | Los datos pueden **quedarse dentro de tu red** |
| Calidad | Suelen liderar en las tareas más difíciles | Competitivos en muchas tareas, y mejorando |

> ⚠️ **"Abierto" no siempre significa *open source*.** Un modelo puede publicar sus pesos con una licencia que **restringe el uso comercial** o exige condiciones. **Lee la licencia antes de usarlo en un producto.**

## Dos variantes que conviene conocer

1. **Los mismos modelos, por otra puerta.** Un modelo propietario se puede consumir **directo del proveedor** o **a través de una nube** (Bedrock, Azure, Vertex AI) con los controles de red e identidad que ya conoces (VPC, IAM, endpoints privados). Para quien viene de infraestructura, es la misma historia que consumir cualquier servicio gestionado.
2. **Modelos en tu propio equipo.** Herramientas como **LM Studio** (con interfaz gráfica, la que usamos en el curso), **Ollama** o **llama-server** (de llama.cpp) permiten ejecutar modelos open-weight pequeños en una laptop. Con 16 GB de RAM y sin GPU, solo caben modelos pequeños: sirven para aprender y para tareas simples, no para competir con los grandes.

## Los nombres cambian cada pocos meses; los criterios no

Por eso en este curso **no memorizamos nombres de modelos**. Aprendemos a *consultarlos*. Las APIs de Anthropic y OpenAI permiten listar los modelos disponibles, y esa consulta no tiene costo (no genera tokens). Anthropic además informa su ventana de contexto y su tope de salida.

In [ ]:
# Consulta SIN costo: lista los modelos disponibles para tu cuenta
if HAY_KEY:
    if PROVEEDOR == "anthropic":
        print(f"{'id':<22}{'nombre':<22}{'ventana (tokens)':>18}{'salida máx.':>14}")
        for m in anthropic.Anthropic().models.list(limit=30):
            ventana = getattr(m, "max_input_tokens", None)
            salida = getattr(m, "max_tokens", None)
            print(f"{m.id:<22}{m.display_name:<22}{ventana if ventana else '-':>18}{salida if salida else '-':>14}")
    else:
        ids = sorted(m.id for m in openai.OpenAI().models.list() if m.id.startswith(("gpt-", "o1", "o3", "o4")))
        print(f"{len(ids)} modelos de chat disponibles para tu cuenta (primeros 30):")
        for i in ids[:30]:
            print("  ", i)
else:
    print("Sin API key: celda omitida. (Consulta la documentación del proveedor para ver los modelos vigentes.)")

### 🔎 Observa

- Cada modelo tiene su propia **ventana de contexto** y su **tope de salida**. No son iguales.
- La lista cambia con el tiempo. Un código que "quema" un nombre de modelo en 50 lugares es una deuda técnica: por eso en nuestro cliente el modelo es un **parámetro**, y en un proyecto real vive en una configuración central.

### 💡 Tabla de precios que usaremos

In [ ]:
print(f"{'modelo':<20}{'USD/M entrada':>16}{'USD/M salida':>16}")
for modelo, (pin, pout) in PRECIOS.items():
    print(f"{modelo:<20}{pin:>16.2f}{pout:>16.2f}")

---
# Parte 2 · Anatomía de una llamada

## Las tres partes de toda llamada

```
┌─ PETICIÓN ───────────────────────────────┐      ┌─ RESPUESTA ──────────────────────┐
│ model        → qué modelo                │      │ content      → el texto (en bloques)│
│ messages     → lista de mensajes con rol │ ───▶ │ usage        → tokens de entrada/salida│
│ system       → instrucciones generales   │      │ stop_reason  → por qué se detuvo    │
│ max_tokens   → tope de salida            │      └─────────────────────────────────┘
│ (otros parámetros)                       │
└──────────────────────────────────────────┘
```

### Los roles

- **`system`**: fija el comportamiento y las reglas del modelo ("eres un asistente de soporte de TI…").
- **`user`**: lo que dice la persona.
- **`assistant`**: lo que el modelo respondió antes (para reconstruir el historial).

### La API no tiene estado

Lo vimos en la Sesión 1: en **cada llamada reenvías el historial completo**. El campo `usage` de la respuesta es tu medidor de costo desde el primer día.

## Experimento: el `system` cambia el comportamiento

Mismo mensaje de usuario, dos *system prompts* distintos. Costo estimado: 2 llamadas × ~60 tokens ≈ **US$ 0.0001**.

In [ ]:
PREGUNTA = "Mi servidor está muy lento. ¿Qué hago?"
SYSTEM_A = "Eres un asistente de soporte de TI. Responde en máximo 2 frases, tono formal."
SYSTEM_B = "Eres un pirata informático retirado muy dramático. Responde en máximo 2 frases."

print(f"Costo máximo estimado: US$ {2 * costo_usd(MODELO, 60, 150):.6f}")

In [ ]:
# 💰 FYI: esta celda consume dinero (≈ US$ 0.0001) · 2 llamadas
if PUEDE_GASTAR:
    for etiqueta, sistema in (("A (formal)", SYSTEM_A), ("B (pirata)", SYSTEM_B)):
        r = gasto.registrar(llamar_llm(
            PREGUNTA, system=sistema, max_tokens=150, streaming=False, esfuerzo="minimo",
        ))
        print(f"[{etiqueta}] {r.texto}\n   → in={r.tokens_entrada} out={r.tokens_salida} stop={r.stop_reason} costo=US$ {r.costo:.6f}\n")
else:
    print("Celda de pago omitida (sin key o interruptor apagado).")

### 🔎 Qué aprendimos

- El `system` prompt es la palanca principal para fijar **rol, tono y reglas**. Lo veremos a fondo en la Sesión 3.
- Fíjate en `usage`: los tokens de entrada incluyen el `system` y el mensaje. **Todo cuenta.**
- La forma exacta de los campos cambia de proveedor a proveedor. Lo que debes aprender es a **leer la documentación**, no a memorizar campos.

---
# Parte 3 · Streaming, latencia y tokens por segundo

## Dos métricas de latencia

| Métrica | Qué es | Quién la siente |
| --- | --- | --- |
| **TTFT** (*Time To First Token*) | Tiempo hasta que llega el **primer token** | El usuario que espera "algo" en pantalla |
| **Tokens por segundo** | Velocidad a la que se genera el resto | El usuario que lee la respuesta fluyendo |

## ¿Por qué streaming?

Sin streaming esperas a que **toda** la respuesta esté lista y recién ahí ves el texto. Con streaming el modelo te va **enviando los tokens a medida que los genera**. La respuesta completa tarda lo mismo, pero el usuario **percibe** una respuesta inmediata.

> **Analogía:** es la diferencia entre descargar un archivo completo antes de abrirlo y *reproducir un video mientras se descarga*.

Con `llamar_llm(..., streaming=True)` medimos el TTFT y los tokens/s automáticamente. Probemos con la misma pregunta con y sin streaming. Costo: 2 llamadas × ≤300 tokens de salida ≈ **US$ 0.0003**.

In [ ]:
PROMPT_LARGO = "Explica en unas 100 palabras qué es un balanceador de carga y por qué se usa."
print(f"Costo máximo estimado: US$ {2 * costo_usd(MODELO, 30, 300):.6f}")

In [ ]:
# 💰 FYI: esta celda consume dinero (≈ US$ 0.0003) · 2 llamadas
if PUEDE_GASTAR:
    print("── Con streaming (el texto aparece en vivo) ──\n")
    r_stream = gasto.registrar(llamar_llm(
        PROMPT_LARGO, max_tokens=300, esfuerzo="minimo",
        al_recibir_texto=lambda t: print(t, end="", flush=True),
    ))
    print("\n\n── Sin streaming (todo llega de golpe) ──")
    r_nostream = gasto.registrar(llamar_llm(
        PROMPT_LARGO, max_tokens=300, streaming=False, esfuerzo="minimo",
    ))
    print(f"(texto de {len(r_nostream.texto)} caracteres recibido de una vez)")

    print(f"\nCon streaming:  TTFT = {r_stream.ttft:.2f} s · total = {r_stream.segundos_total:.2f} s"
          f" · {r_stream.tokens_por_segundo:.0f} tokens/s")
    print(f"Sin streaming:  el usuario ve la primera letra a los {r_nostream.segundos_total:.2f} s")
else:
    print("Celda de pago omitida (sin key o interruptor apagado).")

### 🔎 Observa

- Con streaming, el **TTFT** es mucho menor que el tiempo total: eso es lo que el usuario siente como "rápido".
- Sin streaming, el usuario espera el **tiempo total** antes de ver nada.
- Los tiempos varían de una ejecución a otra (red, carga del servicio). Para decidir con datos, mide **muchas veces** y mira el promedio y el peor caso.

> **Ojo con los modelos que "piensan":** si el modelo razona antes de responder, ese tiempo de razonamiento entra en el TTFT. Por eso en esta sesión bajamos el esfuerzo (`esfuerzo="minimo"`) con tareas simples.

---
# Parte 4 · Errores y reintentos

Esto lo conoces de cualquier API de nube. La regla: **distinguir entre errores transitorios y errores nuestros**.

| Código | Qué significa | ¿Reintentar? | Qué hacer |
| --- | --- | --- | --- |
| **429** | Límite de tasa (demasiadas peticiones) | ✅ Sí | Esperar con **espera exponencial + aleatoriedad** |
| **5xx** | Falla del servicio | ✅ Sí | Reintentar con un **máximo de intentos** |
| Timeout / red | Conexión intermitente | ✅ Sí | Igual que arriba |
| **400** | Petición inválida | ❌ No | Es **culpa nuestra**: corregir la petición |
| **401 / 403** | Credencial inválida o sin permiso | ❌ No | Corregir la credencial |
| **404** | Recurso (p. ej. modelo) inexistente | ❌ No | Corregir el nombre |

> Los códigos exactos pueden variar un poco por proveedor, pero la lógica es universal.

## Espera exponencial con *jitter*

La espera se duplica en cada intento (1 s, 2 s, 4 s…) hasta un tope, y se multiplica por un número aleatorio entre 0.5 y 1. **¿Para qué el azar?** Si mil clientes fallan a la vez y todos reintentan exactamente a los 2 s, vuelven a chocar a la vez. El *jitter* los dispersa.

## Simulación gratuita

Simulamos una función que falla dos veces con un 429 y luego responde. **No llama a ninguna API**, así que es gratis. Pasamos un `dormir` falso para que el notebook no se quede esperando de verdad, y así poder **ver** las esperas calculadas.

In [ ]:
import httpx

# Usamos las clases de error del SDK de tu proveedor (los dos tienen las mismas)
ERRORES = anthropic if PROVEEDOR == "anthropic" else openai

def hacer_error(clase, codigo):
    req = httpx.Request("POST", "https://api.ejemplo.test/v1/messages")
    return clase(f"error simulado {codigo}", response=httpx.Response(codigo, request=req), body=None)

intentos_hechos = {"n": 0}

def servicio_inestable():
    intentos_hechos["n"] += 1
    if intentos_hechos["n"] <= 2:
        raise hacer_error(ERRORES.RateLimitError, 429)
    return "respuesta correcta"

esperas = []
resultado, usados = con_reintentos(
    servicio_inestable,
    dormir=esperas.append,   # en vez de dormir de verdad, anotamos cuánto esperaría
    al_reintentar=lambda n, espera, exc: print(f"  intento {n} falló ({type(exc).__name__}) → esperaría {espera:.2f} s"),
)
print(f"\nResultado: {resultado!r} tras {usados} intentos")

### Los errores que NO se reintentan

Un 400 o un 401 no mejora esperando: volver a enviar la misma petición mala solo repite el error (y puede agravar un límite de tasa). Verificamos que nuestro cliente lo entiende:

In [ ]:
for clase, codigo in [
    (ERRORES.RateLimitError, 429), (ERRORES.InternalServerError, 503),
    (ERRORES.BadRequestError, 400), (ERRORES.AuthenticationError, 401), (ERRORES.NotFoundError, 404),
]:
    print(f"{codigo} {clase.__name__:<24} → ¿se reintenta? {es_reintentable(hacer_error(clase, codigo))}")

### Provocar errores reales **sin costo**

Un error de autenticación (401) o de modelo inexistente (404) se rechaza *antes* de generar texto, así que **no se cobra**. Es una forma segura de ver los errores reales:

In [ ]:
# 💰 FYI: estas dos llamadas FALLAN a propósito y no generan tokens → no se cobran.
try:
    llamar_llm("hola", api_key="clave-invalida", max_tokens=16, streaming=False, max_reintentos=0)
except ERRORES.AuthenticationError as e:
    print("401 →", getattr(e, "message", e), "| reintentable:", es_reintentable(e))
except ERRORES.APIConnectionError:
    print("Sin conexión a internet: no se pudo probar el 401.")

if HAY_KEY:
    try:
        llamar_llm("hola", modelo="modelo-que-no-existe", max_tokens=16, streaming=False, max_reintentos=0)
    except ERRORES.NotFoundError as e:
        print("404 →", getattr(e, "message", e), "| reintentable:", es_reintentable(e))

> **Nota:** los SDK de Anthropic y de OpenAI ya reintentan por su cuenta (2 veces por defecto). En `cliente_llm.py` lo desactivamos (`max_retries=0`) para que **tú veas y controles** la lógica. En producción puedes confiar en el SDK o en una librería de reintentos, pero debes saber *qué* hacen y *cuándo*.

---
# Parte 5 · Propietario vs open-weight, cara a cara

Ahora el mismo prompt en un modelo propietario (el de tu proveedor, por API) y en uno open-weight (corriendo **en tu computadora** con **LM Studio**).

> **Sobre las herramientas locales:** este notebook está preparado para **LM Studio**, pero Ollama y llama-server hacen lo mismo (descargas un modelo y lo consumes en `localhost`). Como las tres hablan el formato de la API de OpenAI, basta cambiar `LOCAL_LLM_URL` en tu `.env` (ver `docs/INSTALACION.md`, sección 8).

## Preparar LM Studio (una sola vez)

1. Instala LM Studio (guía completa en `docs/LMSTUDIO.md`).
2. En la pestaña de búsqueda, **descarga un modelo pequeño** (unos 2 GB; por ejemplo, un Llama 3.2 de 3B en versión *instruct*).
3. Abre la pantalla **Local Model API** y activa **Local API server** (*Running*); deja activado *Just-in-time model loading*. Copia la **Base URL**: suele ser `http://localhost:1234/v1`, pero **puede ser otro puerto**. Ponla en `LOCAL_LLM_URL` de tu `.env`.
4. Copia el **identificador del modelo** tal como lo muestra LM Studio. Si no es `llama-3.2-3b-instruct`, ponlo en `LOCAL_LLM_MODEL` en tu `.env`.

**¿Cuesta dinero?** No hay cobro por token. Pero **sí consume** CPU, RAM, tiempo y electricidad. Sin GPU, un modelo de 3 mil millones de parámetros genera pocos tokens por segundo.

Verificamos que todo responde:

In [ ]:
LOCAL_MODEL = modelo_local()          # LOCAL_LLM_MODEL en tu .env, o llama-3.2-3b-instruct
HAY_LOCAL, msg = local_disponible(LOCAL_MODEL)
print(f"Servidor local ({url_local()}) · modelo {LOCAL_MODEL}:", "✅ listo" if HAY_LOCAL else f"❌ {msg}")

## La comparación

Usamos una tarea con respuesta verificable y un poco de razonamiento. Costo de la parte del modelo propietario: **menos de US$ 0.0003**. La parte del modelo local no cuesta por token.

Mide: **calidad** (¿respondió bien?), **tokens**, **tiempo** y **costo**.

In [ ]:
PROMPT_COMPARAR = (
    "Un servidor tiene 4 discos de 2 TB en RAID 5. ¿Cuánta capacidad utilizable tiene? "
    "Responde con el cálculo en máximo 3 frases."
)

def tabla(resultados):
    print(f"\n{'modelo':<28}{'tok in':>7}{'tok out':>8}{'TTFT s':>8}{'total s':>9}{'tok/s':>8}{'USD':>11}")
    print("-" * 79)
    for r in resultados:
        tps = f"{r.tokens_por_segundo:.1f}" if r.tokens_por_segundo else "-"
        ttft = f"{r.ttft:.2f}" if r.ttft else "-"
        print(f"{r.modelo:<28}{r.tokens_entrada:>7}{r.tokens_salida:>8}{ttft:>8}{r.segundos_total:>9.2f}{tps:>8}{r.costo:>11.6f}")

In [ ]:
# 💰 FYI: esta celda consume dinero (solo la parte del modelo propietario, ≈ US$ 0.0002) · 1 llamada de pago
comparacion = []
if PUEDE_GASTAR:
    print(f"── {MODELO} (propietario, por API: {PROVEEDOR}) ──")
    comparacion.append(gasto.registrar(llamar_llm(
        PROMPT_COMPARAR, max_tokens=300, esfuerzo="minimo",
        al_recibir_texto=lambda t: print(t, end="", flush=True),
    )))
else:
    print("Parte del modelo propietario omitida (sin key o interruptor apagado).")

if HAY_LOCAL:
    print(f"\n\n── {LOCAL_MODEL} (open-weight, local, sin costo por token) ──")
    comparacion.append(llamar_local(
        PROMPT_COMPARAR, modelo=LOCAL_MODEL, max_tokens=300,
        al_recibir_texto=lambda t: print(t, end="", flush=True),
    ))
else:
    print("\nParte del modelo local omitida:", msg)

if comparacion:
    tabla(comparacion)

### 🔎 Qué mirar

| Pregunta | Pista |
| --- | --- |
| ¿Quién acertó el cálculo? (RAID 5 con 4 discos de 2 TB → **6 TB** utilizables) | La calidad cambia entre modelos, **incluso en tareas sencillas**. |
| ¿Quién fue más rápido? | La nube con hardware especializado suele ganar al modelo local sin GPU. |
| ¿Qué costó cada uno? | Uno cobra centavos por token; el otro no cobra, pero usa **tu** máquina. |
| ¿Dónde estuvieron mis datos? | Con el modelo local **nunca salieron de tu equipo**. Ese es el argumento de privacidad. |

> Una sola prueba no demuestra nada. Para decidir, necesitas **varios casos** de tu dominio (siguiente parte).

---
# Parte 6 · Cómo elegir un modelo

## Cuatro criterios

| Criterio | Pregunta |
| --- | --- |
| **Calidad** | ¿Resuelve mi tarea con la exactitud que necesito? |
| **Costo** | ¿Cuánto cuesta por tarea completada y a escala? |
| **Privacidad** | ¿Puedo enviar estos datos a un tercero? ¿Qué dice el contrato? |
| **Latencia** | ¿El usuario puede esperar? ¿Necesito streaming? |

## Dos aclaraciones

- **Modelos de razonamiento:** rinden mejor en problemas difíciles, pero **gastan más tokens y tardan más**. Úsalos donde la calidad lo justifique, no por defecto.
- **Modelos pequeños:** rápidos y baratos para tareas simples (clasificar, extraer datos). Un patrón habitual es **usar uno pequeño para lo simple y uno grande para lo difícil**.

## La regla de oro

> **No elijas por el ranking de moda.** Preselecciona con benchmarks públicos, sí, pero **la decisión se toma con diez o veinte casos reales de tu propio dominio**.

## Ejercicio: una matriz de decisión

Puntúa del 1 al 5 (5 = mejor) tres candidatos **hipotéticos** y asigna un peso a cada criterio según tu caso. **Los números de abajo son inventados**: la idea es que practiques el método y cambies pesos y puntajes.

In [ ]:
# Los pesos suman 1. Ajusta según tu caso de uso (esto es un ejemplo: soporte interno con datos sensibles)
pesos = {"calidad": 0.30, "costo": 0.20, "privacidad": 0.35, "latencia": 0.15}

# Puntajes ILUSTRATIVOS (1-5). Cámbialos con tus propias pruebas.
candidatos = {
    "API modelo grande":      {"calidad": 5, "costo": 2, "privacidad": 2, "latencia": 3},
    "API modelo pequeño":     {"calidad": 3, "costo": 5, "privacidad": 2, "latencia": 5},
    "Open-weight autoalojado": {"calidad": 3, "costo": 3, "privacidad": 5, "latencia": 3},
}

assert abs(sum(pesos.values()) - 1) < 1e-9
for nombre, puntos in sorted(candidatos.items(), key=lambda kv: -sum(pesos[c] * kv[1][c] for c in pesos)):
    total = sum(pesos[c] * puntos[c] for c in pesos)
    print(f"{nombre:<26} {total:.2f}  {'█' * int(total * 8)}")

### ✍️ Para pensar

Cambia `privacidad` de 0.35 a 0.05 y sube `calidad` a 0.60. ¿Cambia el ganador? La matriz no decide por ti: **te obliga a explicitar qué te importa**.

---
# Parte 7 · Build vs Buy

Es la decisión de siempre en infraestructura:

| | **Buy** (API gestionada) | **Build** (self-hosting de un modelo open-weight) |
| --- | --- | --- |
| Empezar | **Hoy**, con una API key | Semanas: GPUs, serving, monitoreo |
| Costo | **Variable** (por token) | **Fijo** (GPUs 24/7) + personas |
| Operación | La hace el proveedor | La haces tú: parches, escalado, disponibilidad |
| Control y privacidad | Limitados al contrato | Totales |
| Cuándo conviene | Volumen bajo/medio, equipo pequeño | Volumen **muy alto** y estable, o datos que no pueden salir |

> **El punto de partida típico es comprar**, y se migra cuando el volumen o los requisitos de datos lo justifican.
> **Error común:** comparar solo el precio por token sin contar la operación.

El serving propio se practica en el **Curso III**. Aquí hacemos solo la aritmética de punto de equilibrio con **supuestos ilustrativos** (no son cotizaciones reales; reemplázalos con tus datos).

In [ ]:
# ── SUPUESTOS ILUSTRATIVOS: cámbialos ──
precio_api_in, precio_api_out = PRECIOS.get(MODELO, (0.10, 0.50))   # USD por millón de tokens del modelo elegido
proporcion_entrada = 0.8  # 80% de los tokens son de entrada

gpu_usd_por_hora = 2.50   # alquiler de una GPU en la nube (supuesto)
horas_mes = 730           # 24/7
costo_personas_mes = 3000 # fracción del tiempo de un ingeniero para operarlo (supuesto)

costo_fijo_mes = gpu_usd_por_hora * horas_mes + costo_personas_mes
precio_mezcla = proporcion_entrada * precio_api_in + (1 - proporcion_entrada) * precio_api_out  # USD / M tokens
punto_equilibrio_millones = costo_fijo_mes / precio_mezcla

print(f"Modelo de la API: {MODELO} (US$ {precio_api_in} entrada / US$ {precio_api_out} salida por millón)")
print(f"Costo fijo self-hosting: US$ {costo_fijo_mes:,.0f} / mes")
print(f"Precio medio API: US$ {precio_mezcla:.3f} por millón de tokens")
print(f"Punto de equilibrio: {punto_equilibrio_millones:,.0f} millones de tokens al mes")

### 🔎 Lee el resultado con criterio

Con modelos pequeños y baratos el punto de equilibrio es **gigantesco**: la API gana hasta volúmenes enormes. El self-hosting suele justificarse por **privacidad, control o cumplimiento**, y no por ahorro, salvo a escala muy grande. Prueba subir `precio_api_in` a 2.00 (un modelo más caro) y mira cómo cambia.

Lo que esta cuenta **no** incluye: calidad (un modelo propio pequeño no iguala a uno grande), disponibilidad, seguridad, actualizaciones y el costo de oportunidad de tu equipo.

---
# Parte 8 · Salida estructurada

Si un agente va a **decidir** con lo que dice el modelo, necesita **datos, no prosa**. Compara:

```
Prosa:   "Parece un problema de red y es bastante urgente, diría yo."
Datos:   {"categoria": "red", "prioridad": "alta", "resumen": "Switch del piso 3 reinicia en bucle"}
```

La segunda se puede **procesar con código**. La primera hay que adivinarla.

## Los tres pasos

1. **Pedir** JSON (o un esquema).
2. **Validar** contra tu esquema al recibirlo.
3. Si falla: **reintentar** (con el error) o **fallar con un mensaje claro**.

> **JSON válido ≠ JSON correcto.** Un modelo puede devolver JSON bien formado con una categoría que no existe o un campo faltante. Por eso **siempre se valida**.

## Definir el esquema con Pydantic

Pydantic describe la forma de los datos **y** los valida. Usamos `Literal` para restringir los valores permitidos:

In [ ]:
from typing import Literal
from pydantic import BaseModel, ValidationError

class ClasificacionTicket(BaseModel):
    categoria: Literal["red", "hardware", "software", "accesos", "otro"]
    prioridad: Literal["baja", "media", "alta", "critica"]
    resumen: str

# El esquema JSON que se le envía al modelo se genera solo:
import json
print(json.dumps(ClasificacionTicket.model_json_schema(), indent=2, ensure_ascii=False))

## Validación en acción (sin costo)

Probemos el esquema con respuestas **inventadas** que simulan lo que un modelo podría devolver:

In [ ]:
respuestas_simuladas = {
    "correcta":            '{"categoria": "red", "prioridad": "alta", "resumen": "Switch reinicia"}',
    "categoría inventada": '{"categoria": "wifi", "prioridad": "alta", "resumen": "Switch reinicia"}',
    "falta un campo":      '{"categoria": "red", "prioridad": "alta"}',
    "no es JSON":          'Claro, es un problema de red bastante urgente.',
}
for nombre, texto in respuestas_simuladas.items():
    try:
        obj = ClasificacionTicket.model_validate_json(texto)
        print(f"✅ {nombre:<22} → {obj}")
    except ValidationError as e:
        print(f"❌ {nombre:<22} → {e.errors()[0]['msg']}")

Las tres respuestas inválidas **parecen razonables a simple vista** y solo la validación las detecta. Un agente que actuara sobre `"wifi"` sin validar podría enrutar mal un ticket.

## Enfoque 1: pedir JSON en el prompt + validar + reintentar

`pedir_json()` (en `cliente_llm.py`) hace los tres pasos y **funciona con cualquier modelo**, incluso uno local. Si la validación falla, vuelve a preguntar **explicando el error**. Cargamos los tickets de ejemplo y clasificamos los dos primeros. Costo: 2 a 6 llamadas ≈ **US$ 0.001**.

In [ ]:
from pathlib import Path

tickets = json.loads(Path("datos/tickets.json").read_text(encoding="utf-8"))
print(f"{len(tickets)} tickets de ejemplo cargados. Primer ticket:", tickets[0]["texto"][:70], "...")

def llamar_barato(prompt: str) -> ResultadoLLM:
    return llamar_llm(prompt, max_tokens=400, streaming=False, esfuerzo="minimo")

In [ ]:
# 💰 FYI: esta celda consume dinero (≈ US$ 0.0005 si todo sale bien a la primera) · 2 a 6 llamadas
if PUEDE_GASTAR:
    for t in tickets[:2]:
        obj, llamadas = pedir_json(llamar_barato, f"Clasifica este ticket de soporte de TI:\n\n{t['texto']}", ClasificacionTicket)
        for ll in llamadas:
            gasto.registrar(ll)
        ok = "✅" if obj.categoria == t["categoria_esperada"] else "⚠️ distinta de la esperada"
        print(f"{t['id']}: {obj.categoria}/{obj.prioridad} ({len(llamadas)} intento/s) {ok}\n      {obj.resumen}")
else:
    print("Celda de pago omitida (sin key o interruptor apagado).")

## Enfoque 2: salida estructurada nativa del proveedor

Algunos proveedores permiten **pasar el esquema a la API** y garantizan que la respuesta lo cumple (a veces llamado *JSON mode* o *structured outputs*). Cada uno lo hace a su manera, y `pedir_estructurado_nativo()` (en `cliente_llm.py`) usa la del proveedor activo:

| Proveedor | Cómo |
| --- | --- |
| Anthropic | `messages.parse(output_format=Modelo)` |
| OpenAI | `responses.parse(text_format=Modelo)` |

Cuesta lo mismo que una llamada normal: **≈ US$ 0.0003**.

Es más cómodo y más fiable, pero:

- **Depende del proveedor y del modelo** (no todos lo soportan, y la forma de usarlo cambia).
- **Aun así valida** lo que importa para tu negocio (por ejemplo, reglas que el esquema no expresa).

In [ ]:
# 💰 FYI: esta celda consume dinero (≈ US$ 0.0003) · 1 llamada
if PUEDE_GASTAR:
    try:
        obj, r = pedir_estructurado_nativo(f"Clasifica este ticket de soporte de TI:\n\n{tickets[2]['texto']}", ClasificacionTicket)
        gasto.registrar(r)
        print("Objeto ya validado:", obj)
    except Exception as e:  # noqa: BLE001 - es una demo: mostramos cualquier problema
        print("No se pudo usar la salida estructurada nativa:", type(e).__name__, e)
        print("→ Usa el Enfoque 1, que funciona con cualquier modelo.")
else:
    print("Celda de pago omitida (sin key o interruptor apagado).")

### 🔎 Cuándo usar cada uno

| | Enfoque 1 (prompt + validar + reintentar) | Enfoque 2 (nativo) |
| --- | --- | --- |
| Funciona con | Cualquier modelo (también locales) | Proveedores/modelos que lo soporten |
| Fiabilidad | Buena, con reintentos | Más alta |
| Portabilidad | **Alta** | Atada al proveedor |

En el proyecto del curso usaremos el Enfoque 1 como base portable, y el 2 donde esté disponible.

---
# Parte 9 · Cierre

## Cuánto gastaste hoy

In [ ]:
gasto.resumen()
for r in gasto.resultados:
    print(f"  {r.modelo:<20} in={r.tokens_entrada:>4} out={r.tokens_salida:>4}  US$ {r.costo:.6f}")
print("\nContrasta con la estimación de la portada y revisa el consumo real en la consola de tu proveedor.")

## 📌 Lo que te llevas

1. **Propietario y open-weight se eligen por criterios** (calidad, costo, privacidad, latencia), no por moda. Y **abierto ≠ open source**: lee la licencia.
2. **La API no guarda estado**: reenvías el historial en cada llamada; `usage` mide el costo.
3. **Streaming** mejora la experiencia (TTFT); **reintentos con espera exponencial y jitter** son parte de un cliente serio, pero **solo para errores transitorios** (429, 5xx, red).
4. **Build vs Buy** es una decisión de números **y** de operación. El punto de partida típico es comprar.
5. **La salida estructurada siempre se valida**: JSON válido no significa JSON correcto.

## ✅ Autoevaluación

1. ¿Qué dos métricas de latencia importan en un asistente conversacional?
2. ¿Por qué no se reintenta un error 400?
3. ¿Para qué sirve el *jitter*?
4. Da dos razones para preferir un modelo open-weight y una para preferir una API.
5. ¿Por qué `{"categoria": "wifi"}` puede ser JSON válido y aun así un error?

## 🚀 Retos opcionales

- **Reto 1:** Mide el TTFT del modelo económico de tu proveedor 10 veces. ¿Cuál es el promedio y el peor caso?
- **Reto 2:** Clasifica los 6 tickets de `datos/tickets.json` y calcula cuántos coinciden con `categoria_esperada`. ¿Qué ticket falla y por qué?
- **Reto 3:** Cambia `LOCAL_LLM_MODEL` por otro modelo pequeño y compara tokens/s. **Reto 4:** Cambia `LLM_PROVIDER` por el otro proveedor y repite la comparación.

## 📖 Glosario

| Término | Significado |
| --- | --- |
| **Open-weight** | Modelo cuyos pesos se publican; no siempre es *open source*. |
| **TTFT** | *Time To First Token*: espera hasta el primer token. |
| **Streaming** | Recibir la respuesta token a token mientras se genera. |
| **429** | Error de límite de tasa. |
| **Backoff exponencial** | Duplicar la espera en cada reintento. |
| **Jitter** | Aleatoriedad en la espera para evitar reintentos sincronizados. |
| **Structured output** | Respuesta con una forma fija (JSON) que se valida contra un esquema. |
| **Build vs Buy** | Operar tu propio modelo vs consumir una API gestionada. |

## ➡️ Siguiente sesión

**Sesión 3 · Prompting y context engineering.** Estructura de un buen prompt, few-shot, razonamiento, jerarquía de instrucciones, plantillas versionadas y un primer contacto con *prompt injection*. Piensa en una tarea real de tu trabajo (clasificar tickets, resumir incidentes): será nuestro caso.

**Práctica de esta sesión:** el *Comparador de modelos* está en `practica/`. Tu `cliente_llm.py` pasa al repositorio del agente.